# Pickling objects

Pickling an object means saving it in a binary format (usually a `PKL` file). Unpickling means converting the `PKL` back to a Python object. It can be really useful when some calculations take a lot of time and you do not want to recompute them every time.

In [ ]:
from importlib import resources
import os
from pprint import pprint
from pathlib import Path

from lightwin.config.config_manager import process_config
from lightwin.core.accelerator.accelerator import Accelerator
from lightwin.ui.workflow_setup import run_simulation
from lightwin.util.pickling import MyCloudPickler

Remove previous pickles

In [ ]:
pickle_folder = Path("pickles")
if not pickle_folder.is_dir():
    os.mkdir(pickle_folder)
else:
    for file in Path("pickles").iterdir():
        os.remove(file)

## Manual pickling/unpickling

### Perform first calculation

We start with an example configuration.

In [ ]:
toml_filepath = resources.files("lightwin.data.ads") / "lightwin.toml"

toml_keys = {
    "files": "files",
    "beam": "beam",
    "beam_calculator": "generic_envelope1d",
    "wtf": "generic_wtf",
    "design_space": "fit_phi_s_design_space",
    "plots": "plots_minimal",
}
override = {
    "plots": {
        "kwargs": {"lw": 5},
        "emittance": False,
        "energy": False,
        "envelopes": False,
        "phase": False,
        "twiss": False,
    },
}

config = process_config(toml_filepath, toml_keys, override=override)
fault_scenarios = run_simulation(config)

### Pickle the Accelerator

In [ ]:
pickler = MyCloudPickler()
fault_scenario = fault_scenarios[0]

reference = fault_scenario.ref_acc
ref_pickle_path = reference.pickle(pickler, path=pickle_folder / "reference-accelerator.pkl")

fixed = fault_scenario.fix_acc
fix_pickle_path = fixed.pickle(pickler, path=pickle_folder / "fixed-accelerator.pkl")

del fault_scenario, reference, fixed

### Unpickle the Accelerator

In [ ]:
reference = Accelerator.from_pickle(pickler, ref_pickle_path)
fixed = Accelerator.from_pickle(pickler, fix_pickle_path)

We can quickly check that we recovered the results.

In [ ]:
ax = None
for accelerator in (reference, fixed):
    ax = accelerator.plot("v_cav_mv", marker="o", ax=ax)

## Configure pickling/unpickling from the `TOML`

### First simple scenario

In [ ]:
pickle_paths = {
    "Reference": pickle_folder / "reference.pkl",
    "000001": {
        "Solution": pickle_folder / "solution-000001-3cavs.pkl"
    },
}
override["files"] = {"pickle_paths": pickle_paths}
config = process_config(toml_filepath, toml_keys, override=override)

In [ ]:
fault_scenarios = run_simulation(config)

In [ ]:
fault_scenarios = run_simulation(config)

### Comparing several scenarios

In [ ]:
pickle_paths = {
    "Reference": pickle_folder / "reference.pkl",
    "000001": {
        "Solution": pickle_folder / "solution-000001-4cavs.pkl",
        "Alternative (3 compensating)": pickle_folder / "solution-000001-3cavs.pkl"
    }
}

override["files"] = {"pickle_paths": pickle_paths}
override["wtf"] = {"k": 4}
config = process_config(toml_filepath, toml_keys, override=override)

In [ ]:
fault_scenarios = run_simulation(config)

Clean pickles folder for the next executions

In [ ]:
for file in pickle_folder.iterdir():
    os.remove(file)